# 00 Run the data pipeline

This notebook runs the whole data pipeline with no terminal needed:

1. Checks the packages are installed
2. Downloads the raw data (only if it is not already on your machine)
3. Cleans it and builds the country panel

The actual code lives in `src/download_data.py` and `src/clean_data.py`. This notebook just runs them, so the logic stays in tidy, reusable files and the notebook stays short.

Run it top to bottom with **Run All**.

In [ ]:
# --- Find the project root and make src/ importable ---
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

## Step 1. Check the packages are installed

If this cell shows an error, run the `%pip` line below it once, then restart the kernel (the Restart button at the top of the notebook) and run everything again.

In [ ]:
import importlib

required = ["pandas", "numpy", "matplotlib", "seaborn", "requests", "pyarrow"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print("Missing packages:", missing)
    print("Run the next cell to install them, then restart the kernel.")
else:
    print("All packages installed.")

In [ ]:
# Only needed if the cell above reported missing packages.
# Remove the # at the start of the next line, run this cell, then restart the kernel.
# %pip install -r {ROOT / "requirements.txt"}

## Step 2. Download the raw data

By default this **does not** re-download if the files are already there. That protects your data snapshot: your results stay based on the same version of the data, which the manifest records. Set `REDOWNLOAD = True` only when you deliberately want the latest data from Our World in Data.

In [ ]:
from src import download_data
from src.config import DATA_RAW, MANIFEST, SOURCES

REDOWNLOAD = False

have_all_files = all((DATA_RAW / name).exists() for name in SOURCES) and MANIFEST.exists()

if have_all_files and not REDOWNLOAD:
    print("Raw data already downloaded. Skipping (set REDOWNLOAD = True to refresh).")
else:
    download_data.main()

In [ ]:
# Show what version of the data you are using
import json

manifest = json.loads(MANIFEST.read_text())
print("Downloaded at (UTC):", manifest["downloaded_at_utc"])
for f in manifest["files"]:
    print(f"  {f['file']:<28} {f['bytes'] / 1e6:6.1f} MB   sha256 {f['sha256'][:12]}...")

## Step 3. Clean the data and build the panel

In [ ]:
from src import clean_data

clean_data.main()

Done. Open `01_data_audit.ipynb` next.